# Krino Inference Server
#
# Run a Krino decision model on free Colab/Kaggle GPU and expose it as a public API.
#
# **How to use:**
# 1. Select GPU runtime: Runtime → Change runtime type → T4 GPU
# 2. Run all cells (Runtime → Run all)
# 3. Copy the public URL printed in Step 4
# 4. Paste into the Krino playground at [oaklight.github.io/krino/playground](https://oaklight.github.io/krino/playground)
#
# **Supports:** Colab (free T4) and Kaggle (free T4 x2, 30hrs/week)

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install torch transformers safetensors huggingface_hub gradio

# Install cloudflared for tunnel
import platform, subprocess, os
arch = platform.machine()
if arch == "x86_64":
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
elif arch == "aarch64":
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64"
else:
    raise RuntimeError(f"Unsupported architecture: {arch}")

subprocess.run(["wget", "-q", url, "-O", "/usr/local/bin/cloudflared"], check=True)
os.chmod("/usr/local/bin/cloudflared", 0o755)
print("✓ Dependencies installed")

## Step 2: Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✓ GPU: {gpu} ({vram:.1f} GB VRAM)")
else:
    print("⚠ No GPU detected — inference will be slow. Enable GPU runtime:")
    print("  Colab: Runtime → Change runtime type → T4 GPU")
    print("  Kaggle: Settings → Accelerator → GPU T4 x2")

## Step 3: Load Model

Choose a model below. Ettin-150m is the fastest (~33ms/query) and recommended for free GPU tiers.

In [ ]:
#@title Select Model { run: "auto" }
MODEL = "oaklight/krino-ettin-150m-heads" #@param ["oaklight/krino-ettin-150m-heads", "oaklight/krino-modernbert-base-heads", "oaklight/krino-qwen3-0.6b-heads", "oaklight/krino-qwen3.5-4b-heads", "oaklight/krino-qwen3-reranker-4b-heads", "oaklight/krino-qwen3-reranker-0.6b-heads"]

import importlib.util, json, time
from pathlib import Path
from huggingface_hub import hf_hub_download

print(f"Loading {MODEL}...")
t0 = time.time()

# Download and import krino.py from the model repo
krino_path = hf_hub_download(MODEL, "krino.py")
spec = importlib.util.spec_from_file_location("krino", krino_path)
krino_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(krino_mod)

# Load model
model = krino_mod.KrinoModel.from_pretrained(MODEL)
elapsed = time.time() - t0
print(f"✓ Model loaded in {elapsed:.1f}s")

# Quick sanity check
answer = model.predict(
    state="The movie was terrible",
    question={"type": "noul", "instructions": "Is this negative?"}
)
print(f"✓ Sanity check: noul={answer.get('noul', '?')}")

## Step 4: Start Gradio Server + Public Tunnel

This starts a Gradio server and exposes it via a Cloudflare quick tunnel.
The public URL will be printed below — copy it to the playground.

In [ ]:
import gradio as gr
import subprocess, threading, re, time, json

AVAILABLE_MODELS = {
    "oaklight/krino-ettin-150m-heads": "Ettin-150m (fastest)",
    "oaklight/krino-modernbert-base-heads": "ModernBERT-base",
    "oaklight/krino-qwen3-0.6b-heads": "Qwen3-0.6B",
    "oaklight/krino-qwen3.5-4b-heads": "Qwen3.5-4B",
    "oaklight/krino-qwen3-reranker-4b-heads": "Qwen3-reranker-4B",
    "oaklight/krino-qwen3-reranker-0.6b-heads": "Qwen3-reranker-0.6B",
}

loaded_models = {MODEL: model}

def get_model(model_id):
    if model_id not in loaded_models:
        print(f"Loading {model_id}...")
        path = hf_hub_download(model_id, "krino.py")
        spec = importlib.util.spec_from_file_location(f"krino_{model_id}", path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        loaded_models[model_id] = mod.KrinoModel.from_pretrained(model_id)
        print(f"✓ {model_id} loaded")
    return loaded_models[model_id]

def predict(model_id, state, question_type, instructions, options_text):
    question = {"type": question_type, "instructions": instructions}
    if question_type in ("choice", "score"):
        parsed = {}
        for line in (options_text or "").strip().splitlines():
            line = line.strip()
            if not line:
                continue
            if ":" in line:
                key, desc = line.split(":", 1)
                parsed[key.strip()] = desc.strip()
            else:
                parsed[line] = line
        if not parsed:
            return {"error": f"{question_type} requires at least one option (format: key: description)"}
        question["criteria" if question_type == "choice" else "legend"] = parsed

    m = get_model(model_id)
    t0 = time.perf_counter()
    try:
        answer = m.predict(state=state, question=question)
    except Exception as e:
        return {"error": str(e)}
    answer["latency_ms"] = round((time.perf_counter() - t0) * 1000, 1)
    answer["model"] = AVAILABLE_MODELS.get(model_id, model_id)
    return answer

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Dropdown(choices=list(AVAILABLE_MODELS.keys()), value=MODEL, label="Model"),
        gr.Textbox(label="State / Input Text", lines=3),
        gr.Radio(choices=["noul", "choice", "score"], value="choice", label="Question Type"),
        gr.Textbox(label="Instructions"),
        gr.Textbox(label="Options (key: description, one per line)", lines=4),
    ],
    outputs=gr.JSON(label="Result"),
    title="Krino Inference Server",
    flagging_mode="never",
)

# Start Gradio in a thread
port = 7860
threading.Thread(target=lambda: demo.launch(server_port=port, share=False, quiet=True), daemon=True).start()
time.sleep(3)

# Start cloudflared tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)

# Wait for tunnel URL
public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  ✓ PUBLIC URL: {public_url}")
    print(f"{'='*60}")
    print(f"\nPaste this URL into the Krino playground:")
    print(f"  https://oaklight.github.io/krino/playground")
    print(f"\nOr test directly:")
    print(f"  {public_url}")
else:
    print("⚠ Tunnel failed to start. Check cloudflared output.")
    print("Fallback: use the local Gradio URL above.")

## Step 5: Keep Alive

Run this cell to keep the session active. Stop the cell (■) to shut down.

In [ ]:
import time
try:
    print("Server running. Press Stop (■) to shut down.")
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    tunnel_proc.terminate()
    print("✓ Server stopped")